In [1]:
from pathlib import Path
import sys
import os

PROJECT_ROOT = Path.cwd().parent

import pandas as pd

from data_paths import (
    HUNTERS_PROCESSED_PATH,
    GATHERERS_PROCESSED_PATH,

    IA_PARAGRAPH_PATH,
    
    DATA_DIR,
)

In [2]:
hunters = pd.read_csv(HUNTERS_PROCESSED_PATH)
gatherers = pd.read_csv(GATHERERS_PROCESSED_PATH)
all_participants = pd.concat([hunters, gatherers], ignore_index=True)

In [3]:
paragraph_ia = pd.read_csv(IA_PARAGRAPH_PATH)
paragraph_ia[['participant_id', 'TRIAL_INDEX', 'auxiliary_span_type',
              'IA_FIRST_FIXATION_TIME', 'IA_LAST_FIXATION_TIME',
              'IA_LAST_FIXATION_DURATION']].head()

,participant_id,TRIAL_INDEX,auxiliary_span_type,IA_FIRST_FIXATION_TIME,IA_LAST_FIXATION_TIME,IA_LAST_FIXATION_DURATION
0,l42_2070,1,outside,17071,17071,276
1,l42_2070,1,outside,17375,17375,179
2,l42_2070,1,outside,17573,18785,351
3,l42_2070,1,outside,.,.,.
4,l42_2070,1,outside,17774,19161,207


In [8]:
def compute_reading_times(data, area_col, regions):
    """Per (participant_id, TRIAL_INDEX), for each region in `regions`:
       RT_pure / RT_normalized — based on first/last IA fixation timestamps.
       TFD_pure / TFD_normalized — sum of IA_DWELL_TIME, raw and divided by n_words.
       Normalization is by number of IAs (rows) in that participant-trial-area."""
    rt_cols = ['IA_FIRST_FIXATION_TIME', 'IA_LAST_FIXATION_TIME', 'IA_LAST_FIXATION_DURATION']
    group_keys = ['participant_id', 'TRIAL_INDEX', area_col]

    df = data[group_keys + rt_cols + ['IA_DWELL_TIME']].copy()
    for c in rt_cols + ['IA_DWELL_TIME']:
        df[c] = pd.to_numeric(df[c], errors='coerce')

    fix = df.dropna(subset=['IA_FIRST_FIXATION_TIME', 'IA_LAST_FIXATION_TIME'])

    agg = fix.groupby(group_keys).agg(
        min_first=('IA_FIRST_FIXATION_TIME', 'min'),
        max_last=('IA_LAST_FIXATION_TIME', 'max'),
    ).reset_index()

    last_idx = fix.loc[fix.groupby(group_keys)['IA_LAST_FIXATION_TIME'].idxmax()]
    last_dur = last_idx[group_keys + ['IA_LAST_FIXATION_DURATION']].rename(
        columns={'IA_LAST_FIXATION_DURATION': 'last_fix_duration'}
    )
    agg = agg.merge(last_dur, on=group_keys)
    agg['RT'] = agg['max_last'] - agg['min_first'] + agg['last_fix_duration']

    tfd = df.groupby(group_keys)['IA_DWELL_TIME'].sum().reset_index(name='TFD')
    word_counts = data.groupby(group_keys).size().reset_index(name='n_words')

    rt_pure = agg.pivot_table(
        index=['participant_id', 'TRIAL_INDEX'], columns=area_col, values='RT',
    ).reset_index()
    tfd_pure = tfd.pivot_table(
        index=['participant_id', 'TRIAL_INDEX'], columns=area_col, values='TFD',
    ).reset_index()
    n_words = word_counts.pivot_table(
        index=['participant_id', 'TRIAL_INDEX'], columns=area_col, values='n_words',
    ).reset_index()

    all_pairs = data[['participant_id', 'TRIAL_INDEX']].drop_duplicates()
    rt_pure = all_pairs.merge(rt_pure, on=['participant_id', 'TRIAL_INDEX'], how='left')
    tfd_pure = all_pairs.merge(tfd_pure, on=['participant_id', 'TRIAL_INDEX'], how='left')
    n_words = all_pairs.merge(n_words, on=['participant_id', 'TRIAL_INDEX'], how='left')

    for wide in (rt_pure, tfd_pure, n_words):
        for col in regions:
            if col not in wide.columns:
                wide[col] = 0
        wide[regions] = wide[regions].fillna(0)

    out = all_pairs.copy()
    for col in regions:
        denom = n_words[col].replace(0, pd.NA).values
        out[f'RT_pure_{col}'] = rt_pure[col].values
        out[f'RT_normalized_{col}'] = pd.Series(rt_pure[col].values / denom).fillna(0).values
        out[f'TFD_pure_{col}'] = tfd_pure[col].values
        out[f'TFD_normalized_{col}'] = pd.Series(tfd_pure[col].values / denom).fillna(0).values
    return out


In [9]:
reading_times = compute_reading_times(
    all_participants,
    area_col='area_label',
    regions=['question', 'answer_A', 'answer_B', 'answer_C', 'answer_D'],
)
reading_times

,participant_id,TRIAL_INDEX,RT_pure_question,RT_normalized_question,TFD_pure_question,TFD_normalized_question,RT_pure_answer_A,RT_normalized_answer_A,TFD_pure_answer_A,TFD_normalized_answer_A,...,TFD_pure_answer_B,TFD_normalized_answer_B,RT_pure_answer_C,RT_normalized_answer_C,TFD_pure_answer_C,TFD_normalized_answer_C,RT_pure_answer_D,RT_normalized_answer_D,TFD_pure_answer_D,TFD_normalized_answer_D
0,l42_2070,4,165.0,41.250000,165.0,41.250000,4744.0,593.000000,1814.0,226.750000,...,1211.0,173.000000,5157.0,736.714286,1030.0,147.142857,3010.0,430.000000,1461.0,208.714286
33,l42_2070,5,0.0,0.000000,0.0,0.000000,1273.0,106.083333,1108.0,92.333333,...,2126.0,236.222222,1309.0,261.800000,771.0,154.200000,1296.0,259.200000,526.0,105.200000
73,l42_2070,6,155.0,11.923077,155.0,11.923077,2596.0,324.500000,1717.0,214.625000,...,656.0,93.714286,5346.0,594.000000,1546.0,171.777778,376.0,94.000000,350.0,87.500000
114,l42_2070,7,142.0,23.666667,142.0,23.666667,4049.0,404.900000,1576.0,157.600000,...,562.0,112.400000,450.0,45.000000,397.0,39.700000,481.0,120.250000,465.0,116.250000
149,l42_2070,8,217.0,31.000000,217.0,31.000000,1353.0,123.000000,912.0,82.909091,...,1401.0,155.666667,3101.0,387.625000,580.0,72.500000,764.0,63.666667,671.0,55.916667
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
760422,l2_324,61,178.0,16.181818,178.0,16.181818,3100.0,442.857143,753.0,107.571429,...,606.0,75.750000,556.0,92.666667,475.0,79.166667,583.0,72.875000,546.0,68.250000
760462,l2_324,62,117.0,16.714286,117.0,16.714286,1260.0,180.000000,546.0,78.000000,...,56.0,8.000000,111.0,18.500000,111.0,18.500000,480.0,96.000000,440.0,88.000000
760494,l2_324,63,169.0,56.333333,164.0,54.666667,400.0,100.000000,400.0,100.000000,...,338.0,48.285714,853.0,142.166667,769.0,128.166667,776.0,97.000000,679.0,84.875000
760522,l2_324,64,186.0,12.400000,147.0,9.800000,956.0,79.666667,802.0,66.833333,...,663.0,60.272727,608.0,60.800000,535.0,53.500000,577.0,57.700000,515.0,51.500000


In [6]:
paragraph_reading_times = compute_reading_times(
    paragraph_ia,
    area_col='auxiliary_span_type',
    regions=['outside', 'distractor', 'critical'],
)
paragraph_reading_times

,participant_id,TRIAL_INDEX,RT_pure_outside,RT_normalized_outside,TFD_pure_outside,TFD_normalized_outside,RT_pure_distractor,RT_normalized_distractor,TFD_pure_distractor,TFD_normalized_distractor,RT_pure_critical,RT_normalized_critical,TFD_pure_critical,TFD_normalized_critical
0,l42_2070,1,14075.0,299.468085,7675.0,163.297872,5710.0,173.030303,4109.0,124.515152,6171.0,128.562500,4655.0,96.979167
128,l42_2070,3,20387.0,224.032967,10079.0,110.758242,14699.0,587.960000,3184.0,127.360000,14035.0,438.593750,5761.0,180.031250
276,l42_2070,4,14032.0,177.620253,8325.0,105.379747,395.0,98.750000,370.0,92.500000,4937.0,329.133333,3144.0,209.600000
374,l42_2070,5,8342.0,238.342857,3614.0,103.257143,3476.0,579.333333,1529.0,254.833333,10548.0,527.400000,4641.0,232.050000
435,l42_2070,6,4801.0,266.722222,4112.0,228.444444,1153.0,115.300000,1034.0,103.400000,5784.0,144.600000,4891.0,122.275000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2631648,l2_324,72,578.0,52.545455,479.0,43.545455,1141.0,28.525000,904.0,22.600000,844.0,49.647059,622.0,36.588235
2631716,l2_324,73,2956.0,64.260870,1876.0,40.782609,462.0,30.800000,368.0,24.533333,2001.0,45.477273,1471.0,33.431818
2631821,l2_324,74,8134.0,88.413043,4962.0,53.934783,142.0,71.000000,142.0,71.000000,1330.0,83.125000,631.0,39.437500
2631931,l2_324,75,6161.0,80.012987,2658.0,34.519481,1263.0,24.764706,947.0,18.568627,2164.0,67.625000,1583.0,49.468750


In [11]:
rt_and_tfd = reading_times.merge(
    paragraph_reading_times,
    on=['participant_id', 'TRIAL_INDEX'],
    how='inner',
)
out_path = DATA_DIR / 'RT_and_TFD.csv'
rt_and_tfd.to_csv(out_path, index=False)
print(f'Saved {len(rt_and_tfd)} rows to {out_path}')
rt_and_tfd

Saved 19436 rows to E:\QA PROJECT\programming\QA_eyetracking_workspace\data\RT_and_TFD.csv


,participant_id,TRIAL_INDEX,RT_pure_question,RT_normalized_question,TFD_pure_question,TFD_normalized_question,RT_pure_answer_A,RT_normalized_answer_A,TFD_pure_answer_A,TFD_normalized_answer_A,...,TFD_pure_outside,TFD_normalized_outside,RT_pure_distractor,RT_normalized_distractor,TFD_pure_distractor,TFD_normalized_distractor,RT_pure_critical,RT_normalized_critical,TFD_pure_critical,TFD_normalized_critical
0,l42_2070,4,165.0,41.250000,165.0,41.250000,4744.0,593.000000,1814.0,226.750000,...,8325.0,105.379747,395.0,98.750000,370.0,92.500000,4937.0,329.133333,3144.0,209.600000
1,l42_2070,5,0.0,0.000000,0.0,0.000000,1273.0,106.083333,1108.0,92.333333,...,3614.0,103.257143,3476.0,579.333333,1529.0,254.833333,10548.0,527.400000,4641.0,232.050000
2,l42_2070,6,155.0,11.923077,155.0,11.923077,2596.0,324.500000,1717.0,214.625000,...,4112.0,228.444444,1153.0,115.300000,1034.0,103.400000,5784.0,144.600000,4891.0,122.275000
3,l42_2070,7,142.0,23.666667,142.0,23.666667,4049.0,404.900000,1576.0,157.600000,...,11358.0,246.913043,21850.0,1456.666667,3672.0,244.800000,9107.0,206.977273,6057.0,137.659091
4,l42_2070,8,217.0,31.000000,217.0,31.000000,1353.0,123.000000,912.0,82.909091,...,7020.0,76.304348,0.0,0.000000,0.0,0.000000,4853.0,303.312500,1819.0,113.687500
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19431,l2_324,61,178.0,16.181818,178.0,16.181818,3100.0,442.857143,753.0,107.571429,...,2121.0,64.272727,128.0,42.666667,128.0,42.666667,3351.0,104.718750,2366.0,73.937500
19432,l2_324,62,117.0,16.714286,117.0,16.714286,1260.0,180.000000,546.0,78.000000,...,2287.0,71.468750,1846.0,63.655172,1319.0,45.482759,2047.0,97.476190,1179.0,56.142857
19433,l2_324,63,169.0,56.333333,164.0,54.666667,400.0,100.000000,400.0,100.000000,...,7760.0,71.192661,4214.0,383.090909,1613.0,146.636364,5679.0,135.214286,2776.0,66.095238
19434,l2_324,64,186.0,12.400000,147.0,9.800000,956.0,79.666667,802.0,66.833333,...,5877.0,59.969388,611.0,87.285714,525.0,75.000000,6059.0,137.704545,4121.0,93.659091
